# Strands Agents 2 - Tools, Memory & MCP

Now we make agents remember well and reach beyond themselves: agent state the model can't see, conversation managers for long chats, sessions that survive restarts, and external tools over the Model Context Protocol (MCP).

**Series:** these notebooks go scratch -> foundations -> intermediate. Run them in order, top to bottom. Every cell here is real, runnable code (verified against `strands-agents` 1.42).

## 0 · What you need (one-time)

1. **Python 3.10+**.
2. An **AWS account** with **Amazon Bedrock model access** enabled for the Claude models we use. In the AWS console: *Bedrock -> Model access -> Enable* **Claude Haiku 4.5** (and optionally **Claude Sonnet 4**). Access is per-region.
3. **AWS credentials** with permission to call Bedrock (`bedrock:InvokeModel` and `bedrock:InvokeModelWithResponseStream`).

You will set the region and credentials two cells down. Nothing else to install by hand - the next cell does it.

### Install the SDK

`%pip` installs into the *kernel that is running this notebook*, so it works the same in Google Colab and in a local VS Code / Jupyter venv.

In [1]:
%pip install -q strands-agents strands-agents-tools mcp
print("installed")

Note: you may need to restart the kernel to use updated packages.
installed


### Configure region + credentials

This cell works in **both** environments:

- **Google Colab:** open the **Secrets** panel (the key icon in the left sidebar), add `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY` (and `AWS_SESSION_TOKEN` if you use temporary keys), then run the cell.
- **VS Code / local:** you usually do not need to paste keys here at all - if you have already run `aws configure` or exported `AWS_` variables in your shell, Strands picks them up automatically. The cell just sets your region.

> Set `AWS_DEFAULT_REGION` to the region where you enabled model access.

In [2]:
import os
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"   # <-- match your enabled-model region

try:
    from google.colab import userdata           # only exists on Colab
    os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
    # os.environ["AWS_SESSION_TOKEN"]   = userdata.get("AWS_SESSION_TOKEN")  # if temporary creds
    print("Colab: credentials loaded from Secrets.")
except ModuleNotFoundError:
    print("Local: using your existing AWS credentials (aws configure / env vars).")
print("Region:", os.environ["AWS_DEFAULT_REGION"])

Local: using your existing AWS credentials (aws configure / env vars).
Region: us-east-1


### Choose a model

Strands talks to Bedrock through a `BedrockModel`. Two things to notice:

- **The `us.` prefix.** Newer Claude models are served on-demand only through a *cross-region inference profile*. The id therefore starts with `us.` (or `global.` / `eu.`). Plain `anthropic.claude-...` would raise *"on-demand throughput isn't supported"*.
- **We set timeouts + adaptive retries** on the underlying boto client. The `adaptive` retry mode automatically backs off and retries when Bedrock throttles you - the single most common error under classroom / burst load.

We default to **Haiku 4.5** because it is fast and cheap, which is ideal while learning. Swap in **Sonnet 4** for harder reasoning by changing one argument.

In [12]:
from strands import Agent, tool, ToolContext
from strands.models import BedrockModel
from botocore.config import Config

MODEL_HAIKU  = "us.anthropic.claude-haiku-4-5-20251001-v1:0"   # fast + cheap (default)
MODEL_SONNET = "us.anthropic.claude-sonnet-4-20250514-v1:0"    # stronger reasoning

def get_model(model_id=MODEL_HAIKU, temperature=0.3, max_tokens=2048):
    """Build a configured Bedrock model. Reuse this everywhere."""
    return BedrockModel(
        model_id=model_id,
        temperature=temperature,        # 0 = deterministic, higher = more varied
        max_tokens=max_tokens,          # cap on the model's reply length
        boto_client_config=Config(
            read_timeout=900, connect_timeout=900,
            retries={"max_attempts": 4, "mode": "adaptive"},   # auto-handle throttling
        ),
    )

MODEL = get_model()

# Small helper: pull clean text out of an AgentResult.
# result.message looks like {"role": "assistant", "content": [{"text": "..."}]}
def say(result):
    return "".join(block.get("text", "") for block in result.message["content"]).strip()

print("Model ready:", MODEL.get_config()["model_id"])

Model ready: us.anthropic.claude-haiku-4-5-20251001-v1:0


### Smoke test (your first real Bedrock call)

If your access, credentials, and region are correct, this prints **setup OK**. If it errors, read the message:

- *AccessDenied / could not load credentials* -> credentials/region not set (cell above).
- *on-demand throughput isn't supported* or *model id ... not found* -> your account exposes this model under a different inference-profile prefix; try `global.` instead of `us.`, or run `aws bedrock list-inference-profiles` to see what is available.

In [4]:
smoke = Agent(model=MODEL, callback_handler=None)   # callback_handler=None => no streaming print
print(say(smoke("Reply with exactly: setup OK")))

setup OK


---
## 1 · A small, real toolkit

Tools are just functions. Here is a tiny "notes" toolkit the agent can use together. Notice each tool does one job and has a clear docstring.

In [5]:
NOTES = []   # stands in for a database

@tool
def add_note(text: str) -> str:
    """Save a short note. Use when the user wants something remembered."""
    NOTES.append(text)
    return f"Saved note #{len(NOTES)}."

@tool
def list_notes() -> str:
    """List all saved notes."""
    return "\n".join(f"{i+1}. {n}" for i, n in enumerate(NOTES)) or "No notes yet."

notes_agent = Agent(model=MODEL, tools=[add_note, list_notes], callback_handler=None)
notes_agent("Note that the demo is on Friday, and that I prefer Haiku for cost.")
print(say(notes_agent("Read my notes back to me.")))

Here are your saved notes:

1. Demo is on Friday
2. Prefer Haiku for cost


## 2 · Agent state: memory the model never sees

`agent.state` is a key-value store that is **not** sent to the model. Use it for things like a user id, a shopping cart, or feature flags - data your *tools* need but that should not cost tokens or leak into the prompt.

A tool reads/writes it through a special parameter named exactly **`tool_context`**. Strands injects the agent there automatically; the model never sees that parameter.

In [ ]:
@tool(
    description="Add an item to the user's shopping cart. The input is the item name.",
    context=True
)
def add_to_cart(item: str, tool_context: ToolContext) -> str:
    """Add an item to the user's cart."""
    cart = tool_context.agent.state.get("cart") or []
    cart.append(item)
    tool_context.agent.state.set("cart", cart)
    return f"Cart now has {len(cart)} item(s)."

shop = Agent(model=MODEL, tools=[add_to_cart], state={"cart": []}, callback_handler=None)
print("Reply-1: ", shop("Add a keyboard and a mouse to my cart."))
print("State (not seen by the model):", shop.state.get("cart"))

print("Reply-2:", say(shop("How many items are in my cart?")))
print("State (not seen by the model):", shop.state.get("cart"))

Reply-1:  Perfect! I've added both items to your cart:
- ✓ Keyboard
- ✓ Mouse

Your cart now has 2 items.

State (not seen by the model): ['keyboard', 'mouse']
Reply-2: Based on the previous actions, your cart has **2 items** - a keyboard and a mouse.
State (not seen by the model): ['keyboard', 'mouse']


## 3 · Conversation managers: keeping long chats in bounds

Every model has a context-window limit. A **conversation manager** decides what history to keep. The default is a **sliding window** that keeps the most recent N messages and drops the oldest (while protecting tool-call pairs).

In [17]:
from strands.agent.conversation_manager import SlidingWindowConversationManager

windowed = Agent(
    model=MODEL,
    conversation_manager=SlidingWindowConversationManager(window_size=6),  # keep last 6 messages
    callback_handler=None,
)
for i in range(4):
    windowed(f"Remember the number {i}.")
print("Messages kept (capped by the window):", len(windowed.messages))
print(say(windowed("What is the most recent number I gave you?")))

Messages kept (capped by the window): 6
The most recent number you gave me is **3**.


## 4 · Summarizing manager: keep the gist of very long chats

When dropping old turns loses too much, the **summarizing** manager compresses older messages into a short summary on overflow and keeps recent ones verbatim. Good for long advisory chats.

In [ ]:
from strands.agent.conversation_manager import SummarizingConversationManager

summarized = Agent(
    model=MODEL,
    conversation_manager=SummarizingConversationManager(
        summary_ratio=0.3,            # fold ~30% of oldest messages when over budget
        preserve_recent_messages=4,   # always keep the last 4 untouched
    ),
    callback_handler=None,
)
print(say(summarized("Hi! I'm planning a 3-city trip: Tokyo, Kyoto, Osaka.")))

# Tokyo, Kyoto, Osaka Trip

Great itinerary! These three cities offer a nice mix of modern and traditional Japan. Here are some quick planning tips:

## Logistics
- **Best order**: Tokyo → Kyoto → Osaka (or reverse)
- **Getting around**: JR Pass might be worth it if you're staying 7+ days
- **Travel times**: 
  - Tokyo to Kyoto: ~2.5 hours (bullet train)
  - Kyoto to Osaka: ~75 minutes (train)

## Quick highlights by city

**Tokyo** - Modern energy, food, nightlife, museums
**Kyoto** - Temples, gardens, traditional culture, geisha districts
**Osaka** - Street food, lively atmosphere, good base for day trips

## Questions to help me give better advice:
- How many days total?
- How many days in each city?
- What's your main interest? (culture, food, nightlife, nature, etc.)
- When are you traveling?
- First time in Japan?

Let me know and I can give you more specific recommendations!


## 5 · Sessions: survive a restart

A **session manager** persists the conversation (and state) to storage automatically - no save/load calls. Build an agent with a `session_manager` and a **required** `agent_id`, then later rebuild it with the same `session_id` to resume.

We run it twice with two *separate* agent objects (simulating two runs) pointed at the same session.

In [19]:
from strands.session.file_session_manager import FileSessionManager

SESSION = "demo-user-001"

# --- run 1 ---
a = Agent(
    model=MODEL, agent_id="assistant",
    session_manager=FileSessionManager(session_id=SESSION, storage_dir="./sessions"),
    callback_handler=None,
)
a("My favorite language is Python.")
print("run 1 done")

# --- run 2: brand new object, SAME session_id => it remembers ---
b = Agent(
    model=MODEL, agent_id="assistant",
    session_manager=FileSessionManager(session_id=SESSION, storage_dir="./sessions"),
    callback_handler=None,
)
print("run 2 recall:", say(b("What is my favorite language?")))

run 1 done
run 2 recall: Your favorite language is Python, as you mentioned in your first message.


**Two things to carry into production:**

- `agent_id` is mandatory with a session manager (it separates agents inside one session). Omit it and you get a `ValueError`.
- `FileSessionManager` has **no locking** - two requests writing the same `session_id` at once can corrupt the file. It is perfect for local development. For a concurrent web app, switch to `S3SessionManager` (atomic writes) or serialize requests per session:

```python
from strands.session.s3_session_manager import S3SessionManager
mgr = S3SessionManager(session_id=SESSION, bucket="my-sessions-bucket")
```

## 6 · MCP: plugging in external tool servers

The **Model Context Protocol (MCP)** is a standard way to expose tools from a separate program (a "server"). Thousands exist (databases, search, GitHub, AWS docs). Strands consumes any MCP server as a tool list.

We will build a tiny MCP server, then connect to it. Two syntax notes:

- **`lambda: ...`** - `MCPClient` wants a *function* that opens the connection, not the connection itself. `lambda` is a one-line anonymous function, so `lambda: stdio_client(...)` hands over "how to connect" without connecting yet.
- **`sys.executable`** - we launch the server with the *same* Python that runs this notebook, so it has the same packages. (Hardcoding `"python"` can pick the wrong interpreter.)

First, write the server to a file:

In [20]:
server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("unit-converter")

@mcp.tool(description="Convert kilometers to miles.")
def km_to_miles(km: float) -> float:
    return round(km * 0.621371, 4)

@mcp.tool(description="Convert miles to kilometers.")
def miles_to_km(miles: float) -> float:
    return round(miles / 0.621371, 4)

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''
with open("unit_server.py", "w") as f:
    f.write(server_code)
print("wrote unit_server.py")

wrote unit_server.py


Now connect. `MCPClient` runs inside a `with` block (a **context manager**): the connection opens at `with` and closes automatically at the end, even on error. You must use the tools while inside the block.

In [ ]:
import sys
from mcp import stdio_client, StdioServerParameters
from strands.tools.mcp import MCPClient

unit_client = MCPClient(lambda: stdio_client(
    StdioServerParameters(command=sys.executable, args=["unit_server.py"])
))

with unit_client:
    mcp_tools = unit_client.list_tools_sync()
    print("Discovered MCP tools:", [t.tool_name for t in mcp_tools])

    converter = Agent(model=MODEL, tools=mcp_tools, callback_handler=None)
    try:
        print(say(converter("How many miles is a 10 km run? Then convert that back to km.")))
    except Exception as e:
        print("Error during conversion:", e)

Discovered MCP tools: ['km_to_miles', 'miles_to_km']
A 10 km run is **6.2137 miles**.

When converted back to kilometers, it equals **10.0 km** (as expected, since we're converting back to the original unit).


That is the whole pattern. Swap `unit_server.py` for any real MCP server - the drug-discovery research assistant in Notebook 3's source blog connects to arXiv, PubMed, and ChEMBL servers exactly this way.

---
## Recap

Agent state (hidden from the model), sliding-window and summarizing managers, durable sessions (and the file-vs-S3 trade-off), and external tools over MCP.

**Next - Notebook 3:** teams of agents that hand off work, and an end-to-end research assistant.